# When Vectors Are Not Enough [Step 1 - The Questions Embeddings Cannot Answer]

> **MLCourse - Agentic AI - Advanced RAG - Graph RAG**

Every retrieval technique in this track so far shares one assumption: **the
answer lives inside some chunk**. Find the right chunk, hand it to the LLM, done.
Reranking finds it more reliably; query transformation finds it when the wording
differs; contextual retrieval returns enough of it to be useful.

Graph RAG exists for the questions where that assumption is simply false - where
the answer is not written down anywhere, because it must be **assembled from
facts stated in different places**.

Ask *"which characters appear at both the tea party and the trial?"* and no
paragraph in the book contains that answer. Two paragraphs list who is at the
tea party; another lists who testifies at the trial; the answer is the
intersection, and it exists only once someone puts them together.

### 1. Setup

As everywhere in this track, `GROQ_API_KEY` is loaded from `03_agentic_ai/.env`
by walking up the directory tree.

In [1]:
import os
import re
import time
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
from dotenv import load_dotenv


def find_env(start=None):
    """Walk up from the notebook directory until a .env file appears."""
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        candidate = folder / ".env"
        if candidate.exists():
            return candidate
    raise FileNotFoundError("No .env found walking up from " + str(start))


ENV_PATH = find_env()
load_dotenv(ENV_PATH)
DATA_DIR = ENV_PATH.parent / "data"

print("env file :", ENV_PATH)
print("data dir :", DATA_DIR)
print("GROQ_API_KEY present:", bool(os.environ.get("GROQ_API_KEY")))

env file : D:\projects\python\MLCourse\03_agentic_ai\.env
data dir : D:\projects\python\MLCourse\03_agentic_ai\data
GROQ_API_KEY present: True


The model is **Groq** `qwen/qwen3.8-27b`. The free tier is around **8000 tokens
per minute**, and graph extraction is token-hungry, so `ask()` paces itself and
backs off. The documented offline fallback is a local Ollama server:
`ChatOllama(model="llama3.1:8b")` is a drop-in replacement.

In [2]:
from langchain_groq import ChatGroq

GROQ_MODEL = "qwen/qwen3.8-27b"          # verified available on this account
llm = ChatGroq(model=GROQ_MODEL, temperature=0)

THINK_RE = re.compile(r"<think>.*?</think>", re.DOTALL)


def clean(text):
    """Strip any <think>...</think> block a reasoning model may emit."""
    return THINK_RE.sub("", text).strip()


def ask(prompt, retries=4, pause=1.5):
    """Call Groq with exponential backoff. Free tier is roughly 8000 tokens/minute,
    so every loop in these notebooks paces itself and retries on rate limits."""
    delay = 5.0
    for attempt in range(retries):
        try:
            answer = clean(llm.invoke(prompt).content)
            time.sleep(pause)
            return answer
        except Exception as exc:
            if attempt == retries - 1:
                raise
            print(f"  [retry {attempt + 1}] {type(exc).__name__} - sleeping {delay:.0f}s")
            time.sleep(delay)
            delay *= 2


print("Groq model:", GROQ_MODEL)
print("smoke test:", ask("Reply with exactly one word: ready"))

Groq model: qwen/qwen3.8-27b


smoke test: ready


In [3]:
ALICE_PATH = DATA_DIR / "alice.txt"
raw_text = ALICE_PATH.read_text(encoding="utf-8-sig")

paragraphs = [" ".join(p.split()) for p in raw_text.split("\n\n") if len(p.strip()) > 200]
print("paragraphs:", len(paragraphs))

paragraphs: 237


In [4]:
from sentence_transformers import SentenceTransformer
import numpy as np

encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
doc_vectors = encoder.encode(paragraphs, normalize_embeddings=True,
                             batch_size=64, show_progress_bar=False)


def dense_rank(query, top_n=5):
    sims = doc_vectors @ encoder.encode([query], normalize_embeddings=True)[0]
    order = np.argsort(sims)[::-1][:top_n]
    return [(int(i), float(sims[i])) for i in order]


print("vector index:", doc_vectors.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

vector index: (237, 384)


### 2. Three question shapes vector search handles badly

**a) Multi-hop.** The answer requires chaining two or more facts that appear in
different chunks. *"Who owns the cat that told Alice where to find the Hatter?"*
needs: Cheshire Cat -> directs Alice to Hatter, and Cheshire Cat -> belongs to
Duchess. No chunk contains both links.

**b) Aggregation.** The answer is a count, a set, or an intersection over the
whole corpus. *"How many characters does Alice meet?"* cannot be answered by
retrieving five paragraphs, because the answer depends on all of them.

**c) Relational / structural.** The question is about the *shape* of the
relationships rather than their content. *"Which character connects the garden
scenes to the courtroom scenes?"* is a question about paths in a network.

Let us watch vector search fail at all three.

In [5]:
HARD_QUESTIONS = [
    ("multi-hop",
     "Who owns the cat that told Alice where to find the Hatter?"),
    ("aggregation",
     "Which characters are present at both the mad tea party and the trial?"),
    ("relational",
     "What connects the Queen of Hearts to the Mock Turtle?"),
]

for kind, question in HARD_QUESTIONS:
    print("=" * 78)
    print(f"[{kind}] {question}")
    print("=" * 78)
    for doc_id, score in dense_rank(question, top_n=3):
        print(f"  cos={score:.3f} doc_{doc_id}: {paragraphs[doc_id][:110]}...")
    print()

[multi-hop] Who owns the cat that told Alice where to find the Hatter?
  cos=0.672 doc_124: Alice waited a little, half expecting to see it again, but it did not appear, and after a minute or two she wa...
  cos=0.599 doc_155: Alice waited till the eyes appeared, and then nodded. “It’s no use speaking to it,” she thought, “till its ear...
  cos=0.594 doc_56: Alice replied eagerly, for she was always ready to talk about her pet: “Dinah’s our cat. And she’s such a capi...

[aggregation] Which characters are present at both the mad tea party and the trial?
  cos=0.456 doc_205: Alice could see, as well as if she were looking over their shoulders, that all the jurors were writing down “s...
  cos=0.446 doc_207: The first witness was the Hatter. He came in with a teacup in one hand and a piece of bread-and-butter in the ...
  cos=0.443 doc_0: CHAPTER I. Down the Rabbit-Hole CHAPTER II. The Pool of Tears CHAPTER III. A Caucus-Race and a Long Tale CHAPT...

[relational] What connects the Queen

Look at what came back. The retrieved paragraphs are *topically plausible* - they
mention the right characters - but none of them states the answer, because no
paragraph does. The retrieval is not broken; the task is outside what
single-chunk retrieval can express.

### 3. What happens when we let the LLM try anyway

This is the important demonstration. Give the model the vector-retrieved context
and ask the multi-hop question. Watch it either say the context is insufficient,
or - worse - fill the gap with plausible invention.

In [6]:
def vector_rag(question, top_k=4):
    ids = [i for i, _ in dense_rank(question, top_n=top_k)]
    context = "\n\n".join(f"[doc_{i}] {paragraphs[i]}" for i in ids)
    return ids, ask(
        "Answer the question using ONLY the context below. If the context does "
        "not contain enough information, say exactly which fact is missing.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"
    )


kind, question = HARD_QUESTIONS[0]
ids, answer = vector_rag(question)
print("question :", question)
print("retrieved:", ids)
print("\nanswer:")
print(answer)

question : Who owns the cat that told Alice where to find the Hatter?
retrieved: [124, 155, 56, 129]

answer:
Based on the provided context, the specific fact that identifies the owner of the cat that told Alice where to find the Hatter is missing.

While [doc_56] mentions a cat named Dinah who belongs to Alice ("Dinah’s our cat"), the context does not state that Dinah is the cat that told Alice where to find the Hatter. The cat in [doc_124] and [doc_155] is referred to only as "the Cat" or "it," and no ownership is attributed to it in the text.


In [7]:
kind, question = HARD_QUESTIONS[1]
ids, answer = vector_rag(question, top_k=5)
print("question :", question)
print("retrieved:", ids)
print("\nanswer:")
print(answer)

question : Which characters are present at both the mad tea party and the trial?
retrieved: [205, 207, 0, 203, 54]

answer:
Based on the provided context, the only character explicitly mentioned in both the "Mad Tea-Party" (Chapter VII, listed in [doc_0]) and the trial (described in [doc_207]) is the **Hatter**.

*   [doc_0] lists "CHAPTER VII. A Mad Tea-Party".
*   [doc_207] states, "The first witness was the Hatter," during the trial.

While the Hatter is a well-known participant in the Mad Tea-Party in the full story, the provided context does not explicitly state that the Hatter was at the Mad Tea-Party; it only lists the chapter title. However, the Hatter is the only character named in the context who is associated with the trial (as a witness) and is traditionally associated with the tea party chapter. If strict adherence to *only* the text provided is required to prove presence at the tea party, the context is missing the explicit statement that the Hatter was present at the Mad

### 4. The same facts, as a graph

Now the alternative representation. Instead of storing text and searching by
similarity, store **entities as nodes and relationships as edges**, and answer
by *traversing*.

```
   Duchess ---owns--> Cheshire Cat ---directs Alice to--> Mad Hatter
                                                              |
                                                        attends
                                                              v
                                                      mad tea party
```

The multi-hop question that vector search could not answer is now a two-edge
walk. Nothing is being *retrieved* in the similarity sense at all - it is being
*computed*.

In [8]:
import networkx as nx

demo = nx.DiGraph()
demo.add_edge("Cheshire Cat", "Mad Hatter", relation="directs Alice to")
demo.add_edge("Cheshire Cat", "Duchess", relation="belongs to")
demo.add_edge("Mad Hatter", "mad tea party", relation="attends")
demo.add_edge("Alice", "mad tea party", relation="attends")
demo.add_edge("Dormouse", "mad tea party", relation="attends")
demo.add_edge("Mad Hatter", "trial", relation="testifies at")
demo.add_edge("Alice", "trial", relation="testifies at")
demo.add_edge("Knave of Hearts", "trial", relation="stands at")

print("nodes:", demo.number_of_nodes(), " edges:", demo.number_of_edges())

# The multi-hop question, answered by walking two edges.
owners = [t for _, t, d in demo.out_edges("Cheshire Cat", data=True)
          if d["relation"] == "belongs to"]
print("\nQ: who owns the cat that directed Alice to the Hatter?")
print("   Cheshire Cat --directs Alice to--> Mad Hatter   (edge 1 identifies the cat)")
print("   Cheshire Cat --belongs to--> ", owners[0], "  (edge 2 gives the answer)")

nodes: 8  edges: 8

Q: who owns the cat that directed Alice to the Hatter?
   Cheshire Cat --directs Alice to--> Mad Hatter   (edge 1 identifies the cat)
   Cheshire Cat --belongs to-->  Duchess   (edge 2 gives the answer)


In [9]:
# The aggregation question, answered as a set intersection over the graph.
tea = {u for u, v in demo.in_edges("mad tea party")}
trial = {u for u, v in demo.in_edges("trial")}

print("at the tea party:", sorted(tea))
print("at the trial    :", sorted(trial))
print("\nBOTH            :", sorted(tea & trial))
print("\nThis is a set operation, not a similarity search. There is no chunk to "
      "retrieve and no threshold to tune - the answer is exact.")

at the tea party: ['Alice', 'Dormouse', 'Mad Hatter']
at the trial    : ['Alice', 'Knave of Hearts', 'Mad Hatter']

BOTH            : ['Alice', 'Mad Hatter']

This is a set operation, not a similarity search. There is no chunk to retrieve and no threshold to tune - the answer is exact.


### 5. What you give up

Graph RAG is not a strict upgrade. It trades away real strengths:

- **Extraction is lossy and expensive.** Turning text into triples costs one LLM
  call per chunk and throws away every nuance that does not fit
  `(subject, relation, object)`. Tone, hedging, conditionals, timing - all gone.
- **Entity resolution is genuinely hard.** "The Queen", "Queen of Hearts", "Her
  Majesty" must become one node, or your graph quietly splits into disconnected
  pieces and your traversals return nothing.
- **Schema drift.** An unconstrained LLM will emit `owns`, `has`, `possesses`,
  and `is the owner of` as four different relations for one idea.
- **Graphs are bad at "what does this passage say".** For ordinary lookup
  questions, plain vector search is better, cheaper and more faithful to the
  source.

Which is why the honest architecture is **hybrid** - vector search for content,
graph traversal for structure - and that is where this module ends
(notebook 05).

### 6. Neo4j and the production question

Everything in this module runs on **NetworkX**, an in-memory Python graph
library. No database, no Docker, no service to start. That is the right choice
for learning, and it is genuinely sufficient up to tens of thousands of nodes.

The production option is a graph database - **Neo4j** being the common one - with
a real query language (Cypher):

```cypher
MATCH (a:Character)-[:ATTENDS]->(:Event {name: 'mad tea party'}),
      (a)-[:TESTIFIES_AT]->(:Event {name: 'trial'})
RETURN a.name
```

That gives you persistence, indexing, concurrent access and multi-million-node
traversals. It also needs a running server - typically
`docker run -p 7687:7687 neo4j` - which **this course deliberately does not
require**. Nothing in these notebooks depends on Docker or on any external
service; where Neo4j is the better production answer, we say so and show the
Cypher, but the code you run stays in-process.

### 7. Key takeaways

- Vector search assumes the answer sits inside one chunk. **Multi-hop,
  aggregation and relational questions break that assumption.**
- A graph stores entities as nodes and relationships as edges, so those
  questions become traversals and set operations - exact, not approximate.
- The costs are real: lossy extraction, entity resolution, schema drift, and
  poor performance on ordinary content questions.
- NetworkX in-memory is the right tool for learning and for modest graphs;
  Neo4j is the production upgrade and needs a server we do not require here.

Next: [`02_entity_extraction.ipynb`](02_entity_extraction.ipynb) - getting
entities and relations out of raw text with the LLM.